# Assignment 20: Building & Deploying a Recommendation System

**Problem Statement:**  
Build a content-based movie recommendation system starting from raw data preprocessing, performing text vectorization, generating recommendations, and deploying using Streamlit.

**Dataset:** Custom Movie Dataset (movies.csv) with movie title, genre, and description  
**Approach:** Content-Based Filtering using TF-IDF + Cosine Similarity  
**Libraries:** Python, Pandas, scikit-learn, Streamlit

---

## Setup: Install & Import Libraries

In [ ]:
# Install required libraries (run once)
# !pip install pandas scikit-learn streamlit

# Import all required libraries
import pandas as pd                              # For data loading and manipulation
import numpy as np                               # For numerical operations
import re                                        # For regular expressions (text cleaning)
from sklearn.feature_extraction.text import TfidfVectorizer   # To convert text to TF-IDF vectors
from sklearn.metrics.pairwise import cosine_similarity        # To compute similarity between items

print('All libraries imported successfully!')

---
## PART 1 — Data Preprocessing

### Task 1: Load & Understand Dataset

In [ ]:
# ============================================================
# Task 1: Load and Understand the Dataset
# ============================================================

# Load the movie dataset from CSV file
# The file has 3 columns: title, genre, description
df = pd.read_csv('movies.csv')

# Print dataset shape (rows × columns)
print('Dataset Shape:', df.shape)
print('Total movies:', df.shape[0])
print('Total columns:', df.shape[1])

# Print column names
print('\nColumn Names:', list(df.columns))

# Print first 5 rows to understand data structure
print('\nFirst 5 rows:')
df.head()

In [ ]:
# Print other essential dataset details

# Data types of each column
print('Data types:')
print(df.dtypes)

# Check for missing values in each column
print('\nMissing values per column:')
print(df.isnull().sum())

# Show basic info
print('\nDataset Info:')
df.info()

# Identify text columns used for recommendations
print('\n--- Text Columns Used for Recommendations ---')
print('1. genre       → genre/category of the movie (e.g., Action, Drama)')
print('2. description → textual plot summary of the movie')
print('These two will be COMBINED to create the content features for recommendation.')

### Task 2: Text Preprocessing for Recommendation

In [ ]:
# ============================================================
# Task 2: Text Preprocessing
# ============================================================

# Step 1: Handle missing values FIRST
# Replace any NaN (empty) values with an empty string
# This prevents errors during text processing
df['genre'] = df['genre'].fillna('')
df['description'] = df['description'].fillna('')

# Step 2: Combine genre and description into one text feature
# We combine both columns so the recommendation uses BOTH genre and plot
# Genre is repeated to give it more weight (more copies = more influence)
df['combined_text'] = df['genre'] + ' ' + df['genre'] + ' ' + df['description']

print('Combined text created. Example:')
print(df['combined_text'][0])
print()

In [ ]:
# Step 3: Define a function to clean text
def clean_text(text):
    """
    Cleans raw text by:
    1. Converting to lowercase (so 'Movie' and 'movie' are treated the same)
    2. Removing punctuation and special characters
    3. Removing extra whitespace
    """
    # 1. Convert all characters to lowercase
    text = text.lower()
    
    # 2. Remove special characters and punctuation using regex
    # [^a-z0-9 ] means: remove anything that is NOT a letter, digit, or space
    text = re.sub(r'[^a-z0-9 ]', '', text)
    
    # 3. Remove extra whitespace (multiple spaces become one space)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Step 4: Apply the cleaning function to combined_text
# TfidfVectorizer handles stopword removal automatically (stop_words='english')
df['clean_text'] = df['combined_text'].apply(clean_text)

# Show the before and after
print('=== Text Preprocessing Results ===')
for i in range(3):
    print(f'\nMovie: {df["title"][i]}')
    print(f'  Original : {df["combined_text"][i][:80]}...')
    print(f'  Cleaned  : {df["clean_text"][i][:80]}...')

---
## PART 2 — Text Vectorization

### Task 3: Vectorization using TF-IDF

In [ ]:
# ============================================================
# Task 3: Vectorization using TF-IDF
# ============================================================

# TF-IDF = Term Frequency - Inverse Document Frequency
# It converts text into numbers where:
#   - Common words get LOW scores (they appear in many movies)
#   - Unique/important words get HIGH scores (they distinguish movies)

# Create a TF-IDF Vectorizer with these settings:
#   max_features=5000 → keep only top 5000 most important words
#   ngram_range=(1,2)  → consider single words AND pairs of words (bigrams)
#   stop_words='english' → automatically remove common English words like 'the', 'is', 'at'
tfidf = TfidfVectorizer(
    max_features=5000,          # Limit vocabulary size
    ngram_range=(1, 2),         # Use unigrams and bigrams
    stop_words='english'        # Remove English stopwords automatically
)

# Fit and transform: learn vocabulary from data and convert text to matrix
# Each row = one movie, Each column = one word/phrase
# Each value = TF-IDF score of that word in that movie
tfidf_matrix = tfidf.fit_transform(df['clean_text'])

# Print the shape of the matrix
print('TF-IDF Matrix Shape:', tfidf_matrix.shape)
print(f'  → {tfidf_matrix.shape[0]} movies (rows)')
print(f'  → {tfidf_matrix.shape[1]} words/phrases (columns)')

# Show a few feature names (the words in vocabulary)
feature_names = tfidf.get_feature_names_out()
print(f'\nSample words in TF-IDF vocabulary (first 20):')
print(list(feature_names[:20]))

### Task 4: Similarity Computation

In [ ]:
# ============================================================
# Task 4: Compute Cosine Similarity
# ============================================================

# WHY cosine similarity?
# Cosine similarity measures the ANGLE between two vectors (not their length).
# This means it doesn't matter if one movie description is longer than another.
# Value ranges from 0 (completely different) to 1 (identical content).
# It's ideal for text because two movies with the same words should be rated similar
# regardless of how long their descriptions are.

# Compute cosine similarity between ALL pairs of movies
# Result is a square matrix of shape (40 movies × 40 movies)
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Store the similarity matrix
print('Cosine Similarity Matrix Shape:', cosine_sim.shape)
print(f'  → {cosine_sim.shape[0]} × {cosine_sim.shape[1]} (movies × movies)')
print()

# Show a sample of the similarity matrix
sim_df = pd.DataFrame(cosine_sim[:5, :5],
                       index=df['title'][:5],
                       columns=df['title'][:5])
print('Sample similarity scores (first 5 movies):')
print(sim_df.round(3))

print('\nNote: Diagonal values = 1.0 (a movie is 100% similar to itself)')
print('Higher values = more similar content')

---
## PART 3 — Recommendation Logic

### Task 5: Build Recommendation Function

In [ ]:
# ============================================================
# Task 5: Build Recommendation Function
# ============================================================

# Create a mapping from movie title → index in the dataframe
# This lets us quickly find any movie by its name
movie_indices = pd.Series(df.index, index=df['title']).drop_duplicates()

def recommend(movie_name, top_n=5):
    """
    Given a movie name, return top N most similar movies.
    
    Parameters:
        movie_name (str): The title of the movie to base recommendations on
        top_n (int)     : Number of recommendations to return (default 5)
    
    Returns:
        DataFrame with recommended movie titles and similarity scores
    """
    # Step 1: Find the index of the given movie in the dataframe
    if movie_name not in movie_indices:
        return f'Movie "{movie_name}" not found in dataset!'
    
    idx = movie_indices[movie_name]  # Get row index of this movie
    
    # Step 2: Get similarity scores for this movie against ALL other movies
    # cosine_sim[idx] gives us a row of similarity scores (one value per movie)
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Step 3: Sort all movies by similarity score in descending order
    # Higher score = more similar movie
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Step 4: Skip the first result (index 0) because it's the movie itself!
    # Then take top_n recommendations
    sim_scores = sim_scores[1:top_n+1]
    
    # Step 5: Get the movie indices and similarity scores
    movie_idx_list = [i[0] for i in sim_scores]      # List of row indices
    score_list = [round(i[1], 4) for i in sim_scores]  # List of scores
    
    # Step 6: Build and return a nice results DataFrame
    result = pd.DataFrame({
        'Rank': range(1, top_n+1),
        'Movie Title': df['title'].iloc[movie_idx_list].values,
        'Genre': df['genre'].iloc[movie_idx_list].values,
        'Similarity Score': score_list
    })
    
    return result

print('Recommendation function created successfully!')

In [ ]:
# Test the recommendation function with 3 different movies

test_movies = ['The Dark Knight', 'Toy Story', 'Interstellar']

for movie in test_movies:
    print(f'\n{"="*55}')
    print(f'  Recommendations for: "{movie}"')
    print(f'{"="*55}')
    result = recommend(movie, top_n=5)
    print(result.to_string(index=False))

---
## PART 4 — Streamlit App (Task 6)

The Streamlit web app is in **`app.py`** (separate file). It provides:
- A dropdown to select any movie
- A slider to choose number of recommendations (1–10)
- A button to generate recommendations
- A table showing recommended movies with genres and similarity scores

**To run the Streamlit app:**
```bash
streamlit run app.py
```

---
## PART 5 — Git & GitHub (Task 7)

Steps to push this project to GitHub:
```bash
# 1. Initialize git repository
git init

# 2. Add all files
git add .

# 3. Commit with a message
git commit -m "Assignment 20: Movie Recommendation System"

# 4. Add your GitHub repository as remote
git remote add origin https://github.com/YOUR_USERNAME/Task20.git

# 5. Push to GitHub
git push -u origin main
```

Repository must include:
- `app.py`
- `movies.csv`
- `requirements.txt`
- `README.md`

---
## PART 6 — Deployment on Render (Tasks 8 & 9)

Steps to deploy on Render:
1. Go to [https://render.com](https://render.com) and create a free account.
2. Click **New +** → **Web Service** → **Connect GitHub**.
3. Select your repository containing this project.
4. Configure deployment:
   - **Build Command:** `pip install -r requirements.txt`
   - **Start Command:** `streamlit run app.py --server.port $PORT --server.address 0.0.0.0`
5. Click **Create Web Service** and wait for deployment.
6. Once deployed, Render gives you a live URL like: `https://your-app.onrender.com`
7. Test the URL to make sure recommendations work correctly.
8. Include the deployed URL in your submission.

In [ ]:
# Final summary of what was built
print('=== Assignment 20 Complete! ===')
print()
print('Tasks Completed:')
print('  Task 1: Loaded dataset with', len(df), 'movies')
print('  Task 2: Preprocessed text (lowercase, removed punctuation, combined features)')
print('  Task 3: TF-IDF vectorization →', tfidf_matrix.shape, 'matrix')
print('  Task 4: Cosine similarity matrix →', cosine_sim.shape)
print('  Task 5: Recommend function built and tested on 3 movies')
print('  Task 6: Streamlit app → see app.py')
print('  Task 7: Git/GitHub → push this folder to GitHub')
print('  Task 8: Render deployment → follow steps above')
print('  Task 9: Validate deployed URL → paste URL in submission')